# Bert For Summarization
Trong bài thực hành này ta sẽ làm quen với mô hình Bert cho bài toán Text Summarization. 

## Bước 1: Cài đặt môi trường và thư viện

In [ ]:
# Không cần Google Drive; mã nguồn sẽ được clone từ GitHub ở cell dưới.

Mã nguồn gốc của bài thực hành [tại đây](https://github.com/nlpyang/BertSum.git).

Dataset gốc của bài thực hành nằm trên Google Drive: [bertsum_data.zip](https://drive.google.com/file/d/1x0d61LP9UAN389YN00z0Pv-7jQgirVg6/view).

Để notebook tải tự động, ta dùng bản mirror public trên Hugging Face: [codemaivanngu/ta-nlp-bertsum-data](https://huggingface.co/datasets/codemaivanngu/ta-nlp-bertsum-data/resolve/main/bertsum_data.zip).

Tải mã nguồn BertSum vào `/content/BertSum` từ GitHub. Cell dưới cũng patch trực tiếp source vừa clone để tương thích với Colab/Python/PyTorch hiện tại; dữ liệu sẽ được tải riêng từ Hugging Face ở bước 2.1.

In [ ]:
BERTSUM_DIR = "/content/BertSum"

In [ ]:
import os
import shutil

if os.path.exists(BERTSUM_DIR):
  shutil.rmtree(BERTSUM_DIR)
!git clone https://github.com/nlpyang/BertSum.git /content/BertSum

# BertSum upstream khá cũ, nên cần patch source sau khi clone để chạy trên Colab mới.
data_loader_path = os.path.join(BERTSUM_DIR, "src/models/data_loader.py")
with open(data_loader_path, "r", encoding="utf-8") as f:
  source = f.read()

# PyTorch mới không cho phép phép trừ với bool tensor: 1 - (src == 0).
# Dùng toán tử phủ định boolean để tạo mask tương đương.
source = source.replace("mask = 1 - (src == 0)", "mask = ~(src == 0)")
source = source.replace("mask_cls = 1 - (clss == -1)", "mask_cls = ~(clss == -1)")

# PyTorch 2.6 đổi default của torch.load thành weights_only=True.
# Các shard .bert.pt/checkpoint của BertSum là object cũ, cần weights_only=False.
source = source.replace("dataset = torch.load(pt_file)", "dataset = torch.load(pt_file, weights_only=False)")
with open(data_loader_path, "w", encoding="utf-8") as f:
  f.write(source)

train_path = os.path.join(BERTSUM_DIR, "src/train.py")
with open(train_path, "r", encoding="utf-8") as f:
  source = f.read()

# Checkpoint BertSum lưu kèm argparse.Namespace, nên validate/test/train_from
# phải load với weights_only=False trên PyTorch 2.6+.
source = source.replace(
  "torch.load(test_from, map_location=lambda storage, loc: storage)",
  "torch.load(test_from, map_location=lambda storage, loc: storage, weights_only=False)",
)
source = source.replace(
  "torch.load(args.train_from,\n                                map_location=lambda storage, loc: storage)",
  "torch.load(args.train_from,\n                                map_location=lambda storage, loc: storage,\n                                weights_only=False)",
)
with open(train_path, "w", encoding="utf-8") as f:
  f.write(source)

pyrouge_path = os.path.join(BERTSUM_DIR, "src/others/pyrouge.py")
with open(pyrouge_path, "r", encoding="utf-8") as f:
  source = f.read()

# Python mới cảnh báo invalid escape sequence với \d trong string thường.
# Chuyển các ví dụ regex sang dạng escape rõ ràng để notebook sạch warning hơn.
source = source.replace(r"SL.P.10.R.11.SL062003-(\d+).html", r"SL.P.10.R.11.SL062003-(\\d+).html")
source = source.replace(r"SL.P/.10.R.{}.SL062003-(\d+).html", r"SL.P/.10.R.{}.SL062003-(\\d+).html")
source = source.replace(r'"(\d+)" part', r'"(\\d+)" part')
with open(pyrouge_path, "w", encoding="utf-8") as f:
  f.write(source)

Cài đặt các thư viện cần thiết

In [ ]:
!pip install pytorch_pretrained_bert -q
!pip install pyrouge -q
!pip install tensorboardX -q

## Bước 2: Giải nén dữ liệu và quan sát một vài mẫu dữ liệu

### 2.1. Giải nén dữ liệu

In [ ]:
!mkdir -p /content/BertSum/bert_data
!wget -O /content/bertsum_data.zip "https://huggingface.co/datasets/codemaivanngu/ta-nlp-bertsum-data/resolve/main/bertsum_data.zip"
!unzip -o /content/bertsum_data.zip -d /content/BertSum/bert_data/

### 2.2. Quan sát dữ liệu

In [ ]:
import os
os.chdir("/content/BertSum")

In [ ]:
import torch

def get_data(subset):
  assert subset in ["train", "test", "valid"]
  data_path = "bert_data"
  data = []
  for file in sorted(os.listdir(data_path)):
    if subset in file:
      ### YOUR CODE HERE: Load dữ liệu tương ứng trong thư mục bert_data
      ### Gợi ý: dùng torch.load(..., weights_only=False) để tương thích PyTorch mới
      data_from_file = None
      data.extend(data_from_file)
      ### END YOUR CODE HERE
  return data

In [ ]:
train_data = get_data("train")

In [ ]:
print(f"Training data có số mẫu dữ liệu như sau: {len(train_data)}")
print(train_data[0].keys())
print("src:", train_data[0]["src"])
print("labels:", train_data[0]["labels"])
print("segs:", train_data[0]["segs"])
print("clss:", train_data[0]["clss"])
print("src_txt:", train_data[0]["src_txt"])
num_wrd_src = 0
for sent in train_data[0]["src_txt"]:
  num_wrd_src += len(sent.split())
print("Độ dài src_txt: ", num_wrd_src)
print("tgt_txt:", train_data[0]["tgt_txt"])
print("Độ dài tgt_txt: ", len(train_data[0]["tgt_txt"].split(" ")))

In [ ]:
# Xóa biến train_data để làm rỗng bộ nhớ
del train_data

### Phần 3: Huấn luyện mô hình và đánh giá

In [ ]:
!mkdir -p /content/BertSum/checkpoints
!touch /content/BertSum/logs/results.log

In [ ]:
import torch
torch.cuda.get_device_name()

In [ ]:
!mkdir -p /content/BertSum/checkpoints/classifier
!mkdir -p  /content/BertSum/checkpoints/transformer
!mkdir -p /content/BertSum/checkpoints/rnn

Assigment: Huấn luyện mô hình: dùng encoder=classifier

In [ ]:
###YOUR CODE HERE

###END YOUR CODE HERE
#Gợi ý: dùng !python src/train.py -mode train -encoder...

Chạy đoạn code dưới đây để cài đặt package ROUGE để tính hiệu năng cho mô hình

In [ ]:
!git clone https://github.com/andersjo/pyrouge.git
from pyrouge import Rouge155
!pyrouge_set_rouge_path /content/BertSum/pyrouge/tools/ROUGE-1.5.5
!sudo cpan App::cpanminus
!sudo cpanm XML::DOM
!python pyrouge/setup.py install


os.chdir("/content/BertSum/pyrouge/tools/ROUGE-1.5.5/data/WordNet-2.0-Exceptions")
if os.path.isfile("./WordNet-2.0.exc.db"):
  os.remove("./WordNet-2.0.exc.db")
! ./buildExeptionDB.pl ./ ./smart_common_words.txt ./WordNet-2.0.exc.db

if os.path.isfile("../WordNet-2.0.exc.db"):
  os.remove("../WordNet-2.0.exc.db")

os.chdir("/content/BertSum/pyrouge/tools/ROUGE-1.5.5/data")
!ln -s WordNet-2.0-Exceptions/WordNet-2.0.exc.db WordNet-2.0.exc.db

Assigment: Đánh giá mô hình:

In [ ]:
os.chdir("/content/BertSum")
###YOUR CODE HERE

###END YOUR CODE HERE
#Gợi ý: dùng !python src/train.py -mode validate ...

In [ ]:
with open("logs/test_results.log", "r") as f:
  result = f.read()
print(result)

Ta có thể thử nghiệm huấn luyện mô hình với các option encoder khác để kiểm tra hiệu năng (transformer, rnn)